## Overview
In this notebook, we will run DCON on a Solovev ideal example equilibrium and plot the results

In [2]:
# Load in necessary packages
using Pkg
using LinearAlgebra
using HDF5
using Plots
using LaTeXStrings

## Run the code
We will run the main DCON code using the inputs specified in `dcon.toml`, `equil.toml`, and `vac.in`. We output the `euler.h5` file, which is a Julia version of the `euler.bin` file.

In [4]:
# Run DCON in Julia
Pkg.activate("../..")
using Revise, JPEC


  Activating project at `~/Code/JPEC_tmp`


  JPEC - Julia Perturbed Equilibrium Code

Equilibrium file: TKMKR_D3Dlike_default_Hmode.geqdsk
--> Processing EFIT g-file: TKMKR_D3Dlike_default_Hmode.geqdsk
--> Parsed from header: nw=257, nh=257

┌ Info: Forcing hamada coordinate jacobian exponents: power_*
└ @ JPEC.Equilibrium /Users/nlogan/Code/JPEC_tmp/src/Equilibrium/EquilibriumTypes.jl:57



   Magnetic axis found at R = 1.74306, Z = -0.00640
   Inboard separatrix found at R = 1.057797647321493.
   Outboard separatrix found at R = 2.3014488562570796.


┌ Info: Setting psilim via dmlim: initial qlim = 5.331249928729366, dmlim = 0.2
└ @ JPEC.ForceFreeStates /Users/nlogan/Code/JPEC_tmp/src/ForceFreeStates/Sing.jl:106


...

┌ Warning: W inverse matrix was non-Hermitian beyond tolerance at 56 integration step(s)
└ @ JPEC.ForceFreeStates /Users/nlogan/Code/JPEC_tmp/src/ForceFreeStates/FixedBoundaryStability.jl:33


Computing free boundary energies
Least Stable Eigenmode Energies:


┌ Info: Using no wall
└ @ JPEC.Vacuum /Users/nlogan/Code/JPEC_tmp/src/Vacuum/VacuumStructs.jl:226


  Plasma = +8.776e-02 +2.847e-05i
  Vacuum = +1.653e+00 -5.839e-06i
  Total  = +1.740e+00 +2.263e-05i
All free-boundary modes stable for n = 1.
Writing saved data to euler.h5

PERTURBED EQUILIBRIUM START
----------------------------------
Loading forcing data from ./forcing.dat
  Loaded 1 forcing modes
Computing plasma response using resp_index=0 (energy-based inductance)
  Building flux matrix from eigenmodes
  Calculating plasma inductance matrix
  Calculating surface inductance from Green's functions


UndefVarError: UndefVarError: `surf_ind` not defined in `JPEC.PerturbedEquilibrium`
Suggestion: check for spelling errors or missing imports.

In [ ]:
JPEC.main(["./"]) # "./" tells us to obtain inputs and direct outputs to our current folder

## Analyze Outputs
We will now analyze the outputs of the run, the most important of which are located in the `euler.h5` output file

In [ ]:
# Read in the euler.h5 data
eh5 = h5open("euler.h5", "r")
mlow = read(eh5["info/mlow"])
xi_psi = read(eh5["integration/xi_psi"])
psifac = read(eh5["integration/psi"])
wt = read(eh5["vacuum/wt"])
crit = read(eh5["integration/crit"])
psio = read(eh5["equil/psio"])
et = read(eh5["vacuum/et"])
close(eh5)

# scale energy eigenvector matrices
chi1 = 2π*psio
wt = wt*(chi1*1e-3)
println("Done reading euler.h5")

### Plot comparison of xi_psi for a few poloidal mode numbers

In [ ]:
p = plot()
for m in 1:5
    plot!(psifac, imag.(xi_psi[m - mlow + 1, 1, :]), label="m=$m")
end
xlabel!(L"\psi_N")
ylabel!(L"\mathrm{Im}(\xi_\psi)")
title!("Least Stable Eigenmode " * L"\xi_\psi" * " for " * "m=1-5")

### Compare the eigenvectors and eigenvalues of each DCON energy matrix eigenmode
This is analagous to the DCON summary plot creating by OMFIT GPEC

In [ ]:
# I got tired of trying to get the Plots version of this to work, so here's a PyPlot version
using PyPlot

# Axes labels
xlabel = "m"
ylabel = "mode (least to most stable)"

yvals = 1:size(wt, 2)
xvals = (1:size(wt, 1)) .+ (mlow + 1)

# Create figure and grid layout
fig = figure(figsize=(9, 7))
gs = fig.add_gridspec(2, 3, height_ratios=[0.25, 0.75], width_ratios=[0.75, 0.21, 0.04])

# Top-left: Eigenvector amplitude
ax0 = fig.add_subplot(gs[1, 1])
ax0.plot(xvals, abs.(wt[:, 1]), color="blue", marker="o", markersize=3)
ax0.set_ylabel("|Eigenvector|")
ax0.set_xlabel("")
ax0.set_title("Mode 1, eigenvalue = $(round(abs(et[1]), digits=3))")

# Bottom-left: Heatmap
ax1 = fig.add_subplot(gs[2, 1])
im = ax1.imshow(abs.(wt') , aspect="auto", origin="lower",
                cmap="viridis", extent=[xvals[1], xvals[end], yvals[1], yvals[end]])
ax1.set_xlabel(xlabel)
ax1.set_ylabel(ylabel)

# Right middle: Eigenvalue amplitude
ax2 = fig.add_subplot(gs[2, 2])
ax2.plot(abs.(et), yvals, color="red", marker="o", markersize=3)
ax2.set_xlabel("Eigenvalue")
ax2.set_xscale("log")
ax2.set_xlim(0.1 * minimum(abs.(et)), 10 * maximum(abs.(et)))
ax2.set_yticks([])

# Colorbar (bottom right)
cax = fig.add_subplot(gs[2, 3])
cb = fig.colorbar(im, cax=cax)
cb.set_label("|W_t_eigenvector|")

# Display and save figure
display(fig)

### Plot crit (the smallest eigenvalue of $W^{-1}$) versus $\Psi$
If crit changes signs during integration, we know we are unstable to an ideal fixed-boundary instability.

In [ ]:
# Plot crit vs psi
p = plot(psifac, crit, legend=false)
xlabel!(L"\psi_N")
ylabel!(L"crit")
title!("Smallest eigenvalue of " * L"W^{-1}" * "(crit) versus " * L"\psi_N")